# HarmFilter — Model Evaluation (F1 scores & Confusion Matrix)

This notebook reproduces the **training/testing accuracy, per-class precision / recall / F1, and the confusion matrix** for the BiLSTM hate-speech model — the evidence requested by the supervisor.

**How to use (in Google Colab):**
1. Run **Cell 1** (install libraries).
2. Run **Cell 2** (download the public dataset).
3. Run **Cell 3** and, when prompted, **upload your 3 model files**: `bilstm_model.h5`, `tokenizer.pkl`, `lstm_config.pkl`.
4. Run **Cell 4** to see the accuracy and F1 report.
5. Run **Cell 5** to plot the confusion matrix.

> The evaluation reproduces the exact training split (stratified 70/30, `random_state=42`) using your saved tokenizer, so the numbers match the project report.


## Cell 1 — Install libraries


In [ ]:
# Colab already has TensorFlow, pandas, scikit-learn, matplotlib.
# We just ensure NLTK stopwords are available.
import nltk
nltk.download('stopwords')
import tensorflow as tf
print('TensorFlow version:', tf.__version__)


## Cell 2 — Download the public dataset
The English model is trained on the public Davidson *hate speech and offensive language* dataset.


In [ ]:
import urllib.request, os
URL = 'https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv'
urllib.request.urlretrieve(URL, 'labeled_data.csv')
print('Downloaded labeled_data.csv:', os.path.getsize('labeled_data.csv'), 'bytes')


## Cell 3 — Upload your trained model files
When the upload box appears, select these three files from your project folder:
`bilstm_model.h5`  ·  `tokenizer.pkl`  ·  `lstm_config.pkl`


In [ ]:
from google.colab import files
uploaded = files.upload()   # choose bilstm_model.h5, tokenizer.pkl, lstm_config.pkl
print('Uploaded:', list(uploaded.keys()))


## Cell 4 — Evaluate: accuracy + per-class F1
Reproduces the training pipeline exactly and prints train/test accuracy and the full classification report.


In [ ]:
import re, pickle
import numpy as np, pandas as pd
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

STOP = set(stopwords.words('english'))

def preprocess_text(text):
    if pd.isna(text) or text == '':
        return ''
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#', '', text)
    for p in list('?:!.,;'):
        text = text.replace(p, '')
    text = text.replace('\n',' ').replace('\t',' ').replace('    ',' ').replace('"','').replace("'s", '')
    words = [w for w in text.split() if w not in STOP]
    return ' '.join(' '.join(words).split())

# Load data exactly as in training
df = pd.read_csv('labeled_data.csv')
df = pd.DataFrame({'tweet': df['tweet'].tolist(), 'class': df['class'].tolist()})
df['tweet_processed'] = df['tweet'].apply(preprocess_text)
df = df[df['tweet_processed'].str.len() > 0]
print('Samples after preprocessing:', len(df))

# Saved tokenizer + config
with open('tokenizer.pkl','rb') as f: tokenizer = pickle.load(f)
with open('lstm_config.pkl','rb') as f: cfg = pickle.load(f)
max_len = cfg.get('max_sequence_length', 100)

X = pad_sequences(tokenizer.texts_to_sequences(df['tweet_processed']), maxlen=max_len, padding='post', truncating='post')
y = df['class'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print('Train:', len(X_train), ' Test:', len(X_test))

model = load_model('bilstm_model.h5')
names = ['Hate Speech', 'Offensive Language', 'Neither']

def evaluate(X, y, split):
    pred = np.argmax(model.predict(X, verbose=0), axis=1)
    acc = accuracy_score(y, pred)
    print(f'\n===== {split} =====')
    print(f'{split} accuracy: {acc*100:.2f}%')
    print(classification_report(y, pred, target_names=names, digits=4))
    return pred

_ = evaluate(X_train, y_train, 'TRAIN')
test_pred = evaluate(X_test, y_test, 'TEST')


## Cell 5 — Confusion matrix (test set)


In [ ]:
import matplotlib.pyplot as plt
cm = confusion_matrix(y_test, test_pred)
print('Confusion matrix:\n', cm)

fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_yticklabels(names)
ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
ax.set_title(f'Confusion Matrix (Test set, n={len(y_test)})')
thresh = cm.max()/2
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black', fontweight='bold')
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix.png — you can download it from the Files panel on the left.')
